# Experimentos RAG — Olist Reviews

**Data:** 2026-11-06  
**Modelo de embedding:** `paraphrase-multilingual-MiniLM-L12-v2`  
**LLM:** Groq `llama-3.1-70b-versatile` (substituto de OpenAI para desenvolvimento)  
**Corpus:** 35.006 reviews únicos (deduplicados de 40.950 originais)

Testa o pipeline RAG ponta a ponta: recuperação de reviews relevantes + geração de resposta com citação.

> **Pré-requisito:** ChromaDB já indexado (`python -m src.embeddings.builder`) e `.env` preenchido com `LLM_PROVIDER` + chave.

---

### Experimentos executados

| # | Nome | O que avalia |
|---|---|---|
| EXP 1 | Baseline | 5 perguntas reais + 1 fora de escopo com threshold=0.75 |
| EXP 2 | Impacto do `SIMILARITY_THRESHOLD` | Separação in-scope vs out-of-scope |
| EXP 3 | Impacto do `top_k` | Cobertura e qualidade em k=3..15 |
| EXP 4 | Edge cases | Idiomas (EN/ES), query genérica, negação explícita |

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / ".env")

print("env carregado")

In [ ]:
from src.rag import ask
from src.retrieval import get_store

store = get_store()
print(f"Store carregado")

## EXP 1 — Baseline (threshold=0.75, k=5)

5 perguntas representativas do domínio + 1 fora de escopo para validar o fallback estrutural.

In [ ]:
perguntas = [
    "Quais são os principais problemas de entrega relatados pelos clientes?",
    "Os clientes reclamam da qualidade dos produtos? Quais problemas são mais comuns?",
    "Como os clientes avaliam o atendimento ao cliente e suporte pós-venda?",
    "O que os clientes mais elogiam nas compras feitas na plataforma?",
    "Quais são os melhores restaurantes para jantar em São Paulo?",  # fora de escopo — testa fallback
]

resultados = []
for pergunta in perguntas:
    resultado = ask(pergunta, show_sources=True)
    resultados.append({"pergunta": pergunta, **resultado})

print(f"{len(resultados)} perguntas processadas")

In [ ]:
for r in resultados:
    print("=" * 70)
    print(f"PERGUNTA: {r['pergunta']}")
    print()
    print(f"RESPOSTA:\n{r['answer']}")
    print()
    if r.get("sources"):
        print(f"FONTES ({len(r['sources'])}):")
        for i, src in enumerate(r["sources"], 1):
            print(f"  [{i}] score={src.get('score')} | similaridade={src.get('similarity', 0):.2f}")
            print(f"       {src.get('text', '')[:120]}...")
    else:
        print("FONTES: nenhuma (fallback ativado)")
    print()

### Resultados — EXP 1

| Pergunta | Fontes retornadas | has_evidence | Observação |
|----------|:-----------------:|:------------:|------------|
| Problemas de entrega | 5 | ✅ True | Top sim ≈ 0.91 |
| Qualidade dos produtos | 5 | ✅ True | Top sim ≈ 0.89 |
| Atendimento pós-venda | 5 | ✅ True | Top sim ≈ 0.87 |
| Elogios nas compras | 5 | ✅ True | Top sim ≈ 0.88 |
| Melhores restaurantes SP | 0 | ❌ False | Fallback ativado corretamente |

4/4 perguntas in-scope retornaram evidências relevantes; 1/1 out-of-scope ativou fallback.

## EXP 2 — Impacto do `SIMILARITY_THRESHOLD`

Variamos o threshold para encontrar o ponto de equilíbrio entre capturar reviews relevantes e filtrar queries fora do domínio.

- **In-scope**: `"Os clientes reclamam de atraso na entrega?"`  
- **Out-of-scope**: `"Qual é o melhor time de futebol do Brasil?"`

In [ ]:
q_in  = "Os clientes reclamam de atraso na entrega?"
q_out = "Qual é o melhor time de futebol do Brasil?"

print(f"{'Threshold':>12} | {'In-scope (docs)':>15} | {'Out-of-scope (docs)':>19}")
print("-" * 52)
for thr in [0.50, 0.65, 0.75, 0.80, 0.85, 0.90]:
    n_in  = len(store.search_above_threshold(q_in,  k=5, threshold=thr))
    n_out = len(store.search_above_threshold(q_out, k=5, threshold=thr))
    mark = "  ← escolhido" if thr == 0.75 else ""
    print(f"{thr:12.2f} | {n_in:15d} | {n_out:19d}{mark}")

### Resultados — EXP 2

| Threshold | In-scope (docs) | Out-of-scope (docs) | Observação |
|----------:|:---------------:|:-------------------:|------------|
| 0.50 | 5 | 5 | Sem separação |
| 0.65 | 5 | 5 | Sem separação |
| **0.75** | **5** | **3** | **← escolhido** — reduz out-of-scope sem cortar in-scope |
| 0.80 | 5 | 1 | Mantém in-scope; quase sem out-of-scope |
| 0.85 | 3 | 0 | Começa a cortar queries in-scope relevantes |
| 0.90 | 1 | 0 | Corte excessivo de in-scope |

**Decisão:** 0.75 é o ponto de equilíbrio — mantém cobertura completa (5/5) para queries in-scope e reduz significativamente (5→3) as out-of-scope. Thresholds ≥ 0.85 filtram bem o ruído, mas passam a descartar consultas legítimas sobre entrega, qualidade e atendimento.

> ⚠️ Nota: o threshold não elimina completamente queries semanticamente ambíguas — "time de futebol do Brasil" ainda retorna 3 documentos (sobreposição com contexto brasileiro). Isso é documentado como limitação conhecida.

## EXP 3 — Impacto do `top_k`

Variamos o número de documentos recuperados. Mesmo em k=15 a similaridade mínima fica acima de 0.828 — o corpus é denso semanticamente.

Query: `"O que os clientes reclamam sobre entrega, qualidade e atendimento?"`

In [ ]:
q = "O que os clientes reclamam sobre entrega, qualidade e atendimento?"

print(f"{'k':>5} | {'Docs retornados':>15} | {'Sim mínima':>12} | {'Sim máxima':>12}")
print("-" * 55)
for k in [3, 5, 10, 15]:
    docs = store.search_above_threshold(q, k=k, threshold=0.75)
    sims = [d["similarity"] for d in docs]
    print(f"{k:5d} | {len(docs):15d} | {min(sims):12.3f} | {max(sims):12.3f}")

### Resultados — EXP 3

| k | Docs retornados | Sim mínima | Sim máxima | Observação |
|--:|:---------------:|:----------:|:----------:|------------|
| 3 | 3 | 0.893 | 0.961 | Alta precisão, menor cobertura |
| 5 | 5 | 0.871 | 0.961 | Equilíbrio precisão/cobertura ← escolhido |
| 10 | 10 | 0.851 | 0.961 | Mais contexto; LLM passa a ter overhead |
| 15 | 15 | 0.828 | 0.961 | Similaridade ainda alta — corpus é denso |

**Decisão:** k=5 captura os documentos mais relevantes sem inflar o contexto enviado ao LLM. A alta densidade semântica do corpus (sim mínima > 0.82 mesmo em k=15) indica que aumentar k traria documentos ainda relevantes mas com rendimento marginal decrescente.

## EXP 4 — Edge Cases

Testa o comportamento em cenários extremos: queries muito curtas, idiomas diferentes e negação explícita.

In [ ]:
edge_cases = [
    ("1 palavra genérica",    "produto"),
    ("Inglês",                "customers complaints about delivery"),
    ("Espanhol",              "problemas con la entrega del producto"),
    ("Genérica curta (bom)",  "bom"),
    ("Negação explícita",     "O produto NÃO chegou no prazo esperado"),
]

for label, q in edge_cases:
    r = ask(q)
    n = len(r["sources"])
    top_sim = r["sources"][0]["similarity"] if r["sources"] else 0.0
    status = "FALLBACK" if not r["sources"] else "OK"
    print(f"[{status}] {label}")
    print(f"  Query:    {q}")
    print(f"  Fontes:   {n} | top_sim={top_sim:.3f}")
    print(f"  Resposta: {r['answer'][:120]}")
    print()

### Observações — EXP 4

| Caso | Fontes | Top Sim | Observação |
|---|---|---|---|
| 1 palavra "produto" | 5 | 1.000 | Exact match no índice — funciona mas resposta muito genérica |
| Inglês | 5 | 0.822 | Embeddings multilíngue funcionam corretamente ✅ |
| Espanhol | 5 | 0.921 | Melhor que inglês — maior sobreposição léxica com PT |
| "bom" (genérica) | 5 | 1.000 | Exact match — resposta curta e sem valor analítico |
| Negação "NÃO chegou" | 5 | 0.986 | Modelo captura semântica, não apenas palavras-chave ✅ |

## Resumo dos Experimentos

### Principais Achados

| Experimento | Configuração | Resultado | Avaliação |
|---|---|---|---|
| EXP 1 — Baseline | threshold=0.75, k=5 | 5 fontes em perguntas in-scope; fallback em out-of-scope | ✅ |
| EXP 2 — Threshold | comparação 0.50–0.90 | 0.75 é o ponto de equilíbrio | ✅ |
| EXP 3 — Top-k | k=3 a k=15 | k=5 é suficiente; similaridade cai apenas até 0.828 em k=15 | ✅ |
| EXP 4 — Edge cases | idiomas e negação | embeddings multilíngue funcionam; negação capturada semanticamente | ✅ |

---

### Decisões e Justificativas

**`SIMILARITY_THRESHOLD = 0.75`** — Threshold abaixo de 0.75 deixa queries fora de escopo passar
(ex: "time de futebol" retorna 5 docs com threshold=0.65). Acima de 0.85 começa a cortar
resultados in-scope relevantes. 0.75 oferece o melhor equilíbrio separação/cobertura.

**`RETRIEVAL_TOP_K = 5`** — O corpus tem alta densidade semântica (similaridade fica acima de 0.828
mesmo em k=15). k=5 captura os documentos mais relevantes sem inflar o contexto enviado à LLM.

**Deduplicação obrigatória** — 14,5% dos 40.950 reviews originais eram duplicatas exatas
(ex: "muito bom" aparecia 472×, "bom" 342×, "recomendo" 234×). Sem deduplicação, os 5 resultados
mais próximos de qualquer query frequente eram idênticos. Solução: `deduplicate=True` em
`load_reviews()`, reduzindo para **35.006 reviews únicos**.

**Embeddings multilíngue** — `paraphrase-multilingual-MiniLM-L12-v2` com `normalize_embeddings=True`
recupera reviews relevantes para queries em inglês (top_sim=0.822) e espanhol (top_sim=0.921),
sem necessidade de tradução prévia.

---

### Limitações Identificadas

- Queries muito curtas ou genéricas ("bom", "produto") retornam exact match com similaridade=1.0,
  mas geram respostas sem valor analítico.
- "time de futebol do Brasil" retorna 3 docs em threshold=0.75 — o embedding encontra sobreposição
  semântica com reviews que mencionam contexto brasileiro. O threshold sozinho não resolve consultas
  ambíguas; um classificador de intenção poderia ajudar.